In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!pip install hmmlearn scikit-learn

In [ ]:
import tensorflow as tf

print("GPU:", tf.config.list_physical_devices('GPU'))

GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


In [ ]:
import os
import numpy as np
import json
from hmmlearn import hmm
from sklearn.metrics import mean_absolute_error, f1_score

DATA_DIR = "/content/drive/MyDrive/FYP/phase3"

APPLIANCES = ["toaster", "kettle", "computer", "lamp"]

train_houses = list(range(1, 17))   # 1–16
test_houses  = list(range(17, 21))  # 17–20

In [ ]:
def load_data(appliance, houses, max_samples=5000):
    X_all, y_all = [], []

    for file in os.listdir(DATA_DIR):
        if appliance not in file or not file.endswith(".npz"):
            continue

        house_id = int(file.split("_")[0].replace("house", ""))
        if house_id not in houses:
            continue

        path = os.path.join(DATA_DIR, file)
        data = np.load(path)

        print(f"Loading: {file}")

        X = data["X"]
        y = data["y"]

        X = X[:max_samples]
        y = y[:max_samples]

        X_all.append(X)
        y_all.append(y)

    return np.vstack(X_all), np.hstack(y_all)

In [ ]:
results = {}

for app in APPLIANCES:

    print("\n====================")
    print("Processing:", app)
    print("====================")

    # =========================
    # LOAD DATA
    # =========================
    X_train, y_train = load_data(app, train_houses)
    X_test, y_test   = load_data(app, test_houses)

    print("Train:", X_train.shape, "Test:", X_test.shape)

    # =========================
    # FEATURE (mean of window)
    # =========================
    X_train_feat = np.mean(X_train, axis=1).reshape(-1,1)
    X_test_feat  = np.mean(X_test, axis=1).reshape(-1,1)

    # =========================
    # FHMM (Gaussian HMM)
    # =========================
    model = hmm.GaussianHMM(
        n_components=4,
        covariance_type="diag",
        n_iter=100,   # ✅ as requested
        tol=1e-3,
        random_state=42
    )

    model.fit(X_train_feat)

    # =========================
    # PREDICT STATES
    # =========================
    states = model.predict(X_test_feat)

    # =========================
    # STATE → ON/OFF
    # =========================
    means = model.means_.flatten()
    sorted_idx = np.argsort(means)

    state_map = {}
    for i, s in enumerate(sorted_idx):
        state_map[int(s)] = 0 if i < len(sorted_idx)//2 else 1

    y_pred = np.array([state_map.get(int(s), 0) for s in states])

    # =========================
    # GROUND TRUTH
    # =========================
    threshold = np.percentile(y_train, 50)
    y_true = (y_test > threshold).astype(int)

    # =========================
    # EVALUATION
    # =========================
    mae = mean_absolute_error(y_true, y_pred)
    f1  = f1_score(y_true, y_pred)

    print("State means:", means)
    print("State map:", state_map)
    print("MAE:", mae)
    print("F1 Score:", f1)

    results[app] = {"MAE": mae, "F1": f1}

    # =========================
    # SAVE FOR RASPBERRY PI
    # =========================
    save_data = {
        "means": means.tolist(),
        "state_map": state_map,
        "threshold": float(threshold)
    }

    with open(f"/content/{app}_fhmm.json", "w") as f:
        json.dump(save_data, f)


Processing: toaster
Loading: house2_toaster_seq2point.npz
Loading: house3_toaster_seq2point.npz
Loading: house5_toaster_seq2point.npz
Loading: house6_toaster_seq2point.npz
Loading: house7_toaster_seq2point.npz
Loading: house8_toaster_seq2point.npz
Loading: house10_toaster_seq2point.npz
Loading: house12_toaster_seq2point.npz
Loading: house14_toaster_seq2point.npz
Loading: house18_toaster_seq2point.npz
Train: (45000, 599) Test: (5000, 599)


State means: [0.03772849 0.24573195 0.08340938 0.03629437]
State map: {3: 0, 0: 0, 2: 1, 1: 1}
MAE: 0.485
F1 Score: 0.009799918334013884

Processing: kettle
Loading: house2_kettle_seq2point.npz
Loading: house3_kettle_seq2point.npz
Loading: house4_kettle_seq2point.npz
Loading: house5_kettle_seq2point.npz
Loading: house6_kettle_seq2point.npz
Loading: house7_kettle_seq2point.npz
Loading: house8_kettle_seq2point.npz
Loading: house9_kettle_seq2point.npz
Loading: house11_kettle_seq2point.npz
Loading: house12_kettle_seq2point.npz
Loading: house13_kettle_seq2point.npz
Loading: house16_kettle_seq2point.npz
Loading: house18_kettle_seq2point.npz
Loading: house19_kettle_seq2point.npz
Train: (60000, 599) Test: (10000, 599)
State means: [0.03278937 0.23609967 0.05408908 0.09098934]
State map: {0: 0, 2: 0, 3: 1, 1: 1}
MAE: 0.4123
F1 Score: 0.01669449081803005

Processing: computer
Loading: house1_computer_seq2point.npz
Loading: house5_computer_seq2point.npz
Loading: house6_computer_seq2point.npz
Load

In [ ]:
from google.colab import files

for app in APPLIANCES:
    files.download(f"/content/{app}_fhmm.json")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [ ]:
print("\n========= FINAL FHMM RESULTS =========")

for app, res in results.items():
    print(f"\n{app.upper()}")
    print("MAE:", res["MAE"])
    print("F1 :", res["F1"])


========= FINAL FHMM RESULTS =========

TOASTER
MAE: 0.485
F1 : 0.009799918334013884

KETTLE
MAE: 0.4123
F1 : 0.01669449081803005

COMPUTER
MAE: 0.28613333333333335
F1 : 0.6601203674374406

LAMP
MAE: 0.5854
F1 : 0.34774373259052926
